In [6]:
import os
import torch
import torch.nn as nn
import numpy as np
import mlflow
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

mlflow.set_tracking_uri(f"sqlite:///{os.path.abspath('../../mlflow.db')}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])


train_dataset = datasets.CIFAR10(root='../../datasets', train = True, download = False, transform=transform)
test_dataset = datasets.CIFAR10(root='../../datasets', train=False, download=False, transform=transform)


c:\miniconda3\envs\aio\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [8]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(512, 5)

target_classes = torch.tensor([0,1,2,3,4])

train_targets, test_targets = torch.tensor(train_dataset.targets), torch.tensor(test_dataset.targets)
train_indices, test_indices = torch.where(torch.isin(train_targets, target_classes))[0].tolist(), torch.where(torch.isin(test_targets, target_classes))[0].tolist()

train_sub = Subset(train_dataset, train_indices)
test_sub = Subset(test_dataset, test_indices)

train_loader = DataLoader(dataset=train_sub, batch_size=32, shuffle=False)
test_loader = DataLoader(dataset=test_sub, batch_size=32, shuffle=False)

In [9]:
epoch = 5
lr = 0.001
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
model = model.to(device)

mlflow.set_experiment("ResNet18_CIFAR10")

<Experiment: artifact_location='file:c:/Users/kiet0/Desktop/GitHub/ai-ml-learning-portfolio/deep_learning/ResNet18_Transfer_CIFAR10/mlruns/4', creation_time=1781597373421, experiment_id='4', last_update_time=1781597373421, lifecycle_stage='active', name='ResNet18_CIFAR10', tags={}, trace_location=None, workspace='default'>

In [10]:

with mlflow.start_run(run_name="ResNet18_5class"):
    for i in range(epoch):
        model.train()

        train_loss = 0
        train_batches = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            preds = model(images)
            loss = criterion(preds, labels)

            train_loss += loss.item()
            train_batches += 1

            loss.backward()
            optimizer.step()

        model.eval()

        val_loss = 0
        corrects = 0
        samples = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                preds = model(images)
                loss = criterion(preds, labels)
                val_loss += loss.item()
                corrects += (torch.argmax(preds, dim=1) == labels).sum().item()
                samples += labels.size(0)

        avg_train_loss = train_loss / train_batches
        avg_val_loss = val_loss / samples
        val_accuracy = corrects / samples

        mlflow.log_metric("train_loss", avg_train_loss, step=i)
        mlflow.log_metric("val_accuracy", val_accuracy, step=i)
        mlflow.log_metric("avg_val_loss", avg_val_loss, step=i)




    

        

